# 03 — LoRA Fine-Tuning

Standard LoRA experiment with a frozen full-precision/half-precision base model where hardware permits. Keep configuration values shared with the QLoRA notebook.

In [ ]:
import os, json, time, torch, pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
DATA_PATH = "../data/train.jsonl"
OUTPUT_DIR = "../models/lora_adapter"
SEED = 42

## Shared experiment configuration

In [ ]:
CONFIG = {
    "seed": SEED,
    "learning_rate": 2e-4,
    "epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "max_seq_length": 512,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "save_steps": 100,
    "rank": 16,
    "alpha": 32,
    "dropout": 0.05,
}
print(json.dumps(CONFIG, indent=2))

## Load prepared data

In [ ]:
def load_jsonl_dataset(path):
    return load_dataset("json", data_files=path, split="train")

# train_dataset = load_jsonl_dataset(DATA_PATH)
# validation_dataset = load_jsonl_dataset("../data/validation.jsonl")

## Tokenizer and model loading

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_lora_base():
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map="auto"
    )

# model = load_lora_base()

## Build text field

In [ ]:
def format_training_record(record):
    prompt = record.get("instruction", "")
    context = record.get("context", "")
    response = record.get("response", "")
    user_content = prompt if not context else prompt + "\\n\\nContext:\\n" + context
    return {"text": f"### Instruction\\n{user_content}\\n\\n### Response\\n{response}"}

# train_dataset = train_dataset.map(format_training_record)
# validation_dataset = validation_dataset.map(format_training_record)

## Configure LoRA adapters

In [ ]:
lora_config = LoraConfig(
    r=CONFIG["rank"],
    lora_alpha=CONFIG["alpha"],
    lora_dropout=CONFIG["dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
# model = get_peft_model(model, lora_config)
# model.print_trainable_parameters()

## Training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_strategy="steps",
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

## Trainer construction

In [ ]:
# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=validation_dataset,
#     processing_class=tokenizer,
#     dataset_text_field="text",
#     max_seq_length=CONFIG["max_seq_length"],
# )

## Train and record runtime

In [ ]:
# if torch.cuda.is_available():
#     torch.cuda.reset_peak_memory_stats()
# start_time = time.perf_counter()
# train_result = trainer.train()
# elapsed_seconds = time.perf_counter() - start_time
# peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
# metrics = {**train_result.metrics, "elapsed_seconds": elapsed_seconds, "peak_gpu_gb": peak_gpu_gb}
# Path("../results").mkdir(exist_ok=True)
# with open("../results/lora_training_metrics.json", "w") as f:
#     json.dump(metrics, f, indent=2)
# print(metrics)

## Verify saved adapter

In [ ]:
# from pathlib import Path
# adapter_files = list(Path(OUTPUT_DIR).glob("*"))
# assert adapter_files, "No adapter files found; training/save step may not have completed."
# print([p.name for p in adapter_files])

## Experiment record

After running, write down the GPU type, software versions, dataset fingerprint, training configuration, and any errors. Do not compare results from different data splits without noting the difference.

## Verified configuration output

This output is generated by evaluating the `CONFIG` dictionary already present in this notebook. It confirms intended hyperparameters, not a completed training run.

In [1]:
for key, value in CONFIG.items():
    print(f"{key}: {value}")


seed: 42
learning_rate: 0.0002
epochs: 1
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
max_seq_length: 512
warmup_ratio: 0.03
weight_decay: 0.01
logging_steps: 10
save_steps: 100
rank: 16
alpha: 32
dropout: 0.05

## Runtime dependency and hardware check

This is a genuine inspection of the current runtime. It does not attempt to download model weights or start training.

In [2]:
import importlib.util, torch, platform
required = ["transformers", "datasets", "peft", "trl"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Missing training dependencies:", missing)
print("Readiness:", "BLOCKED" if (missing or not torch.cuda.is_available()) else "CHECK GPU MEMORY")


Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA available: False
Missing training dependencies: ['transformers', 'datasets', 'peft', 'trl']
Readiness: BLOCKED

## Training output status

The supplied `trainer.train()` cell is commented out. No training loss or elapsed time can honestly be reported until the training is run.